# Day 5 Tutorial：随机森林与 Bagging

## Goal

复现一次 Bootstrap，公平比较单树与森林，并从内部树预测手工复算森林平均。

## Setup

所有随机过程固定为 42；使用两列人工特征，不访问 test。

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(42)
indices = np.arange(5)
bootstrap_indices = rng.choice(indices, size=len(indices), replace=True)
print('original:', indices)
print('bootstrap:', bootstrap_indices)

original: [0 1 2 3 4]
bootstrap: [0 3 3 2 2]


In [2]:
X_train = np.array([
    [0.0, 0.0], [1.0, 1.0], [2.0, 0.0], [3.0, 1.0],
    [4.0, 0.0], [5.0, 1.0], [6.0, 0.0], [7.0, 1.0],
    [8.0, 0.0], [9.0, 1.0],
])
y_train = np.array([0.2, 1.1, 1.9, 3.2, 3.9, 5.1, 5.8, 7.2, 7.9, 9.1])
X_valid = np.array([[1.5, 0.0], [3.5, 1.0], [5.5, 0.0], [7.5, 1.0]])
y_valid = np.array([1.4, 3.6, 5.4, 7.6])

assert X_train.shape == (10, 2)
print('data shapes:', X_train.shape, y_train.shape, X_valid.shape, y_valid.shape)

data shapes: (10, 2) (10,) (4, 2) (4,)


## Steps

两个模型使用相同数据、深度上限和评价函数。

In [3]:
models = {
    'one_tree': DecisionTreeRegressor(max_depth=3, random_state=42),
    'random_forest': RandomForestRegressor(
        n_estimators=100,
        max_depth=3,
        max_features=1,
        bootstrap=True,
        random_state=42,
        n_jobs=1,
    ),
}

rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    for split, X_split, y_split in [
        ('train', X_train, y_train),
        ('valid', X_valid, y_valid),
    ]:
        prediction = model.predict(X_split)
        rows.append({
            'model': model_name,
            'split': split,
            'mae': float(mean_absolute_error(y_split, prediction)),
            'rmse': float(np.sqrt(mean_squared_error(y_split, prediction))),
            'r2': float(r2_score(y_split, prediction)),
        })

results = pd.DataFrame(rows)
results

,model,split,mae,rmse,r2
0,one_tree,train,0.150000,0.237697,0.993016
1,one_tree,valid,0.237500,0.288314,0.984045
2,random_forest,train,0.504493,0.614198,0.953372
3,random_forest,valid,0.221467,0.230527,0.989800


## Checks

森林回归输出应等于内部树预测的逐样本平均。

In [4]:
forest = models['random_forest']
tree_prediction_matrix = np.vstack([
    tree.predict(X_valid) for tree in forest.estimators_
])
manual_average = tree_prediction_matrix.mean(axis=0)
forest_prediction = forest.predict(X_valid)

assert tree_prediction_matrix.shape == (100, 4)
assert np.allclose(manual_average, forest_prediction)
assert len(np.unique(np.round(tree_prediction_matrix[:, 0], 8))) > 1
print('forest averaging checks passed')
print('first valid prediction:', round(float(forest_prediction[0]), 4))

forest averaging checks passed
first valid prediction: 1.6225


## Next Steps

完成 `03_exercises.md`。个人结果保存到 `experiments/day05_random_forest/`；教程表不能作为自己的运行证据。